# ICU Federated Learning — Data Preparation
1. Loads `vitals.csv`
2. Engineers features
3. **Fits and saves** a `StandardScaler` → `scaler.pkl`
4. Splits the SCALED data across 3 hospital CSVs

> ⚠️ The scaler is fit ONCE here on all data, then saved.
> Clients load the CSV (already scaled) and do NOT re-scale.

In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import os

os.makedirs('data', exist_ok=True)

In [13]:
#Load dataset
df = pd.read_csv('data/vitals.csv')
print('Shape:', df.shape)
print(df.head(2))

Shape: (200020, 17)
   Patient ID  Heart Rate  Respiratory Rate                   Timestamp  \
0           1          60                12  2024-07-19 21:53:45.729841   
1           2          63                18  2024-07-19 21:52:45.729841   

   Body Temperature  Oxygen Saturation  Systolic Blood Pressure  \
0         36.861707          95.702046                      124   
1         36.511633          96.689413                      126   

   Diastolic Blood Pressure  Age  Gender  Weight (kg)  Height (m)  \
0                        86   37  Female    91.541618    1.679351   
1                        84   77    Male    50.704921    1.992546   

   Derived_HRV  Derived_Pulse_Pressure  Derived_BMI  Derived_MAP Risk Category  
0     0.121033                      38    32.459031    98.666667     High Risk  
1     0.117062                      42    12.771246    98.000000     High Risk  


In [14]:
#Drop non-feature columns
drop_cols = [c for c in ['Patient ID', 'Timestamp'] if c in df.columns]
df = df.drop(columns=drop_cols)

In [15]:
#  Encode label 
df['Risk Category'] = df['Risk Category'].str.strip().str.lower()
df['Risk Category'] = df['Risk Category'].map({'high risk': 1, 'low risk': 0})
print('Label distribution:')
print(df['Risk Category'].value_counts())

Label distribution:
Risk Category
1    105115
0     94905
Name: count, dtype: int64


In [16]:
#Encode Gender 
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

In [17]:
# Drop any rows with NaN 
df.dropna(inplace=True)
print('Shape after feature engineering:', df.shape)

Shape after feature engineering: (200020, 15)


In [18]:
#Separate features and label 
X = df.drop('Risk Category', axis=1)
y = df['Risk Category']

feature_columns = list(X.columns)
print('Features:', feature_columns)

Features: ['Heart Rate', 'Respiratory Rate', 'Body Temperature', 'Oxygen Saturation', 'Systolic Blood Pressure', 'Diastolic Blood Pressure', 'Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Derived_HRV', 'Derived_Pulse_Pressure', 'Derived_BMI', 'Derived_MAP']


In [19]:
# Fit scaler on ALL data and SAVE it
# KEY FIX: scaler is fit here ONCE and saved.
# clients load this same scaler for prediction only (not re-training).
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_columns, index=X.index)
# Save scaler for predict.py
joblib.dump(scaler, 'scaler.pkl')
print('scaler.pkl saved')

scaler.pkl saved


In [20]:
# Combine features + label
df_processed = X_scaled.copy()
df_processed['label'] = y.values

# Shuffle
df_processed = df_processed.sample(frac=1, random_state=42).reset_index(drop=True)

print('Final shape:', df_processed.shape)
print('Label balance:')
print(df_processed['label'].value_counts())

Final shape: (200020, 15)
Label balance:
label
1    105115
0     94905
Name: count, dtype: int64


In [21]:
#Split across hospitals (equal thirds)
n = len(df_processed)
hospital_A = df_processed.iloc[:n // 3]
hospital_B = df_processed.iloc[n // 3 : 2 * n // 3]
hospital_C = df_processed.iloc[2 * n // 3:]

hospital_A.to_csv('data/hospitalA.csv', index=False)
hospital_B.to_csv('data/hospitalB.csv', index=False)
hospital_C.to_csv('data/hospitalC.csv', index=False)

print(f'Hospital A: {len(hospital_A)} rows | label dist: {hospital_A["label"].value_counts().to_dict()}')
print(f'Hospital B: {len(hospital_B)} rows | label dist: {hospital_B["label"].value_counts().to_dict()}')
print(f'Hospital C: {len(hospital_C)} rows | label dist: {hospital_C["label"].value_counts().to_dict()}')
print('\nData ready for Federated Learning')

Hospital A: 66673 rows | label dist: {1: 34944, 0: 31729}
Hospital B: 66673 rows | label dist: {1: 35157, 0: 31516}
Hospital C: 66674 rows | label dist: {1: 35014, 0: 31660}

Data ready for Federated Learning


In [22]:
#Verify
df_check = pd.read_csv('data/hospitalA.csv')
print('Sample rows from hospitalA.csv:')
print(df_check.head(3))
print('Columns:', list(df_check.columns))

Sample rows from hospitalA.csv:
   Heart Rate  Respiratory Rate  Body Temperature  Oxygen Saturation  \
0    1.338737         -1.520811          1.274149          -1.685968   
1    0.213475         -1.084980         -1.492203          -0.104208   
2    0.646269         -1.520811         -1.079273          -0.736732   

   Systolic Blood Pressure  Diastolic Blood Pressure       Age    Gender  \
0                 1.104553                 -0.260477 -1.224158 -0.998931   
1                -0.166106                 -1.650033 -0.694975 -0.998931   
2                -1.436766                 -0.260477 -0.310115  1.001070   

   Weight (kg)  Height (m)  Derived_HRV  Derived_Pulse_Pressure  Derived_BMI  \
0    -1.158788   -1.032623     1.720431                1.063118    -0.353696   
1     0.942868    0.042888    -0.337705                0.774793     0.579390   
2    -0.309070    1.326381    -0.791630               -1.051267    -0.977069   

   Derived_MAP  label  
0     0.455950      1  
1    